# Suite SER — Saving and loading an architecture

The library singleton reaches every part of an architecture, so a store's snapshot holds it all, with the schemas it
uses. A channel's message schema is written as the schema object's path, so it is found again in the store that reads
it. A snapshot does not carry the meta-schemas' extension that lets a schema object fill `Interactions.HasSchema`'s
link; `define` adds it to a store that lacks it. A reachable document refers to owned objects (modules, interfaces,
ports) by path, and writes counted ones (implementations, instances, channels, containers, axes) whole.

In [ ]:
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from mbse.Architecture.Systems.Examples import Pipeline
from mbse.Architecture.Systems.Framework import Interactions
from support import raises


def built():
    store = Builders.Store()
    return store, Pipeline.build(store)


def pipeline_in(store):
    """The pipeline module a store's library holds."""
    return store.singleton("Interactions.Library").modules.at("Pipeline").module

## SER-01 · A snapshot read into a bare store, then `define`, is the same architecture, byte for byte

In [ ]:
store, p = built()
text = JSON.ToJSON(store).Snapshot()
copy = Builders.Store()
JSON.FromJSON(copy).Snapshot(text)
I = copy.registered("Interactions.Module")
module = pipeline_in(copy)
# Before define, nothing lets a schema object fill HasSchema's link, so validation reports each channel's.
assert len(Validators.Validate(copy).Reachable(I, module)) == 2
Interactions.define(copy)
assert Validators.Validate(copy).Reachable(I, module) == [] and Interactions.Check()(module) == []
assert JSON.ToJSON(copy).Snapshot() == text  # the extension is no data
impl = next(iter(module.implementations)).impl
assert Interactions.path(impl.channels.at("raw").channel) == "reference.raw"
assert Interactions.message_schema(impl.channels.at("raw").channel) is copy.registered("dsp.Sample")
assert [e.role for e in Interactions.ends(impl.channels.at("filtered").channel)] == ["source", "destination", "destination"]

## SER-02 · Into a store that defined the schemas first, the same; YAML reads back alike

In [ ]:
store, p = built()
text = JSON.ToJSON(store).Snapshot()
copy = Builders.Store()
Interactions.define(copy)
JSON.FromJSON(copy).Snapshot(text)
assert JSON.ToJSON(copy).Snapshot() == text
assert Validators.Validate(copy).Reachable(copy.registered("Interactions.Module"), pipeline_in(copy)) == []

again = Builders.Store()
YAML.FromYAML(again).Snapshot(YAML.ToYAML(store).Snapshot())
Interactions.define(again)
assert JSON.ToJSON(again).Snapshot() == text

## SER-03 · A reachable document writes counted objects whole and refers to owned ones by path

In [ ]:
store, p = built()
document = JSON.ToJSON(store).Reachable(p.schemas.Module, p.pipeline)
assert '"singleton": "Interactions.Library"' in document  # the modules, their interfaces and ports: by path

# Read into the same store, the owned objects are found again; the counted ones are built anew.
read = JSON.FromJSON(store).Reachable(p.schemas.Module, document)
assert read is p.pipeline
assert [Interactions.path(e.impl) for e in p.pipeline.implementations] == ["reference", "reference"]

# A store without the modules has nothing at those paths, and refuses the document, writing none of it.
other = Builders.Store()
Interactions.define(other)
with other.transaction():
    other.Schemas.OfObject(lambda o: o.name("dsp.Sample"))
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    JSON.FromJSON(other).Reachable(other.registered("Interactions.Module"), document)
assert list(other.singleton("Interactions.Library").modules) == []